# 02 - Probability Gap Case Study

This notebook reconstructs the case-study mechanics from the Cytokinetics slide: scenario prices, market-implied probability, model checklist probability and probability gap.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

from biotech_catalyst.checklist_scoring import score_catalyst, adjust_probability_from_score
from biotech_catalyst.implied_probability import expected_value, market_implied_probability, probability_edge
from biotech_catalyst.position_sizing import expected_return_from_scenarios


In [ ]:
case = pd.read_csv(ROOT / "data" / "cytokinetics_case_study.csv")
row = case.iloc[0]
display(case)


In [ ]:
implied_probability = market_implied_probability(
    current_price=row["current_price"],
    approval_price=row["approval_price"],
    failure_price=row["failure_price"],
)

model_probability = row["model_probability"]
gap = probability_edge(model_probability, implied_probability)
expected_catalyst_value = expected_value(
    model_probability,
    row["approval_price"],
    row["failure_price"],
)
expected_return = expected_return_from_scenarios(
    current_price=row["current_price"],
    approval_price=row["approval_price"],
    failure_price=row["failure_price"],
    model_probability=model_probability,
)

case_summary = pd.DataFrame(
    [
        ["current price", row["current_price"]],
        ["approval price", row["approval_price"]],
        ["failure price", row["failure_price"]],
        ["market-implied probability", implied_probability],
        ["model probability", model_probability],
        ["probability gap", gap],
        ["expected catalyst value", expected_catalyst_value],
        ["expected return", expected_return],
    ],
    columns=["metric", "value"],
)

display(case_summary)


In [ ]:
scores = {
    "endpoint_strength": row["endpoint_strength"],
    "safety": row["safety"],
    "regulatory_classification": row["regulatory_classification"],
    "cmc_risk": row["cmc_risk"],
    "standard_of_care": row["standard_of_care"],
}

total_score = score_catalyst(scores)
checklist_probability = adjust_probability_from_score(
    baseline_probability=0.80,
    total_score=total_score,
    mean_score=10,
    penalty_per_point=0.05,
)

checklist_table = pd.DataFrame(
    list(scores.items()) + [("total_score", total_score), ("checklist_probability", checklist_probability)],
    columns=["item", "value"],
)

display(checklist_table)


In [ ]:
probabilities = pd.DataFrame(
    {
        "probability_type": ["market implied", "model checklist"],
        "probability": [implied_probability, checklist_probability],
    }
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(probabilities["probability_type"], probabilities["probability"])
ax.set_ylim(0, 1)
ax.set_ylabel("Approval probability")
ax.set_title("Cytokinetics probability gap")
for idx, value in enumerate(probabilities["probability"]):
    ax.text(idx, value + 0.02, f"{value:.1%}", ha="center")
plt.show()
